# EDA — creditcard.csv
**Objective:** Understand the PCA-anonymised bank credit card transaction dataset, quantify imbalance, and assess feature distributions before modelling.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120


## 1. Load Data

In [ ]:
from src.data_loader import load_creditcard_data

cc_df = load_creditcard_data('../data/raw/creditcard.csv')
print("Shape:", cc_df.shape)
cc_df.head()


## 2. Data Overview

In [ ]:
print("=== Missing Values ===")
print(cc_df.isna().sum().sum(), "total missing values")
print()
print("=== Duplicates ===")
print(cc_df.duplicated().sum(), "duplicate rows")
print()
cc_df.describe()[['Time', 'Amount', 'Class']].T


## 3. Class Imbalance

In [ ]:
class_counts = cc_df['Class'].value_counts()
class_pct = cc_df['Class'].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values,
       color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Class Distribution — creditcard.csv', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Transactions')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 1000, f'{v:,}\n({class_pct.iloc[i]:.3f}%)', ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/class_imbalance_cc.png', bbox_inches='tight')
plt.show()


## 4. Transaction Amount & Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cc_df['Amount'], bins=100, color='steelblue', log=True)
axes[0].set_title('Transaction Amount (log scale)')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count (log)')

axes[1].hist(cc_df['Time'], bins=100, color='teal')
axes[1].set_title('Transaction Time (seconds from first)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig('../reports/figures/amount_time_cc.png', bbox_inches='tight')
plt.show()


## 5. PCA Feature Distributions (V1–V28)

In [ ]:
v_cols = [c for c in cc_df.columns if c.startswith('V')]
fig, axes = plt.subplots(4, 7, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(v_cols):
    cc_df[col].hist(bins=40, ax=axes[i], color='slateblue', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_yticks([])

plt.suptitle('V1–V28 Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/pca_features_cc.png', bbox_inches='tight')
plt.show()


## 6. Bivariate: Amount and Time vs Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, group in cc_df.groupby('Class'):
    axes[0].hist(group['Amount'], bins=80, alpha=0.6,
                  label=f"{'Fraud' if label else 'Legit'}", density=True)
axes[0].set_title('Amount by Class (density)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

for label, group in cc_df.groupby('Class'):
    axes[1].hist(group['Time'], bins=80, alpha=0.6,
                  label=f"{'Fraud' if label else 'Legit'}", density=True)
axes[1].set_title('Time by Class (density)')
axes[1].set_xlabel('Time (s)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/bivariate_cc.png', bbox_inches='tight')
plt.show()


## 7. Correlation Heatmap (Top V Features with Class)

In [ ]:
corr = cc_df[v_cols + ['Amount', 'Class']].corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
corr.plot(kind='barh', ax=ax, color=['tomato' if v < 0 else 'steelblue' for v in corr.values])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Class (Fraud=1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_cc.png', bbox_inches='tight')
plt.show()


## 8. Key EDA Findings

- **Extreme imbalance**: Only ~0.17% of transactions are fraud — roughly 492 out of 284,807.
- **Amount**: Fraudulent transactions tend to be smaller amounts, possibly to avoid detection.
- **Time**: No strong temporal pattern in fraud timing.
- **V4, V11, V14, V17** show the strongest correlation with the fraud label.
- **No missing values** — PCA pre-processing ensures completeness.
- **Duplicate rows** exist and will be dropped in preprocessing.